# 232 Group Project Part 2: Data Exploration

### 1. GitHub Repository Setup
Milestone 2 Branch: https://github.com/rowheaton/ragus-wheaton-iribe-232/tree/milestone2

**Import Packages**

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType, DoubleType, FloatType
from pyspark.sql.functions import col, count, length,countDistinct, broadcast, min as spark_min, max as spark_max,avg, stddev, approx_count_distinct
import requests
import pandas as pd
import matplotlib.pyplot as plt

### 2. SDSC Expanse Environment Setup

8 total cores and 128 GB total memory, reserving 2 GB for the Spark driver. Executor instances = 8 - 1 = 7. Executor memory = (128 - 2) / 7 = 18 GB per executor. Spark is configured with 7 executor instances and 18 GB executor memory.

Calculated executer memory is ~17.7GB but we reduced it to 16 GB and allocated 2 GB memory overhead per executor to prevent memory exhaustion during heavy operations.

In [ ]:
total_cores = 8
total_memory_gb = 128
driver_memory_gb = 4   

executor_instances = total_cores - 1
raw_executor_memory = (total_memory_gb - driver_memory_gb) / executor_instances

executor_memory_gb = 16
executor_overhead_gb = 2

print(f"Executor instances = {executor_instances}")
print(f"Raw executor memory = {raw_executor_memory:.2f} GB")
print(f"Configured executor memory = {executor_memory_gb} GB (+{executor_overhead_gb} GB overhead)")

spark = (
    SparkSession.builder
    .appName("MusicBrainz")
    .config("spark.driver.memory", f"{driver_memory_gb}g")
    .config("spark.executor.instances", executor_instances)
    .config("spark.executor.memory", f"{executor_memory_gb}g")
    .config("spark.executor.memoryOverhead", f"{executor_overhead_gb}g")
    .config("spark.sql.shuffle.partitions", executor_instances * total_cores)
    .getOrCreate()
)

sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

response = requests.get(url)
executors = response.json()

spark_df = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
spark_df['maxMemory_GB'] = (spark_df['maxMemory'] / (1024**3)).round(2)
spark_df

### 3. Data Exploration using Spark

**Define Variables**

In [ ]:
MBDUMP = "/expanse/lustre/projects/uci157/rwheaton1/mbdump"

In [ ]:
def peek(df, n=20):
    df.show(n, truncate=False)

**Define Schemas**

In [ ]:
artist_schema = StructType([
    StructField("id",               IntegerType(),   True),
    StructField("gid",              StringType(),    True),
    StructField("name",             StringType(),    True),
    StructField("sort_name",        StringType(),    True),
    StructField("begin_date_year",  IntegerType(),   True),
    StructField("begin_date_month", IntegerType(),   True),
    StructField("begin_date_day",   IntegerType(),   True),
    StructField("end_date_year",    IntegerType(),   True),
    StructField("end_date_month",   IntegerType(),   True),
    StructField("end_date_day",     IntegerType(),   True),
    StructField("type",             IntegerType(),   True),
    StructField("area",             IntegerType(),   True),
    StructField("gender",           IntegerType(),   True),
    StructField("comment",          StringType(),    True),
    StructField("edits_pending",    IntegerType(),   True),
    StructField("last_updated",     StringType(),    True),
    StructField("ended",            StringType(),    True),
    StructField("begin_area",       IntegerType(),   True),
    StructField("end_area",         IntegerType(),   True),
])
instrument_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("gid",           StringType(),  True),
    StructField("name",          StringType(),  True),
    StructField("type",          IntegerType(), True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
    StructField("comment",       StringType(),  True),
    StructField("description",   StringType(),  True),
])
label_schema = StructType([
    StructField("id",                IntegerType(), True),
    StructField("gid",               StringType(),  True),
    StructField("name",              StringType(),  True),
    StructField("begin_date_year",   IntegerType(), True),
    StructField("begin_date_month",  IntegerType(), True),
    StructField("begin_date_day",    IntegerType(), True),
    StructField("end_date_year",     IntegerType(), True),
    StructField("end_date_month",    IntegerType(), True),
    StructField("end_date_day",      IntegerType(), True),
    StructField("label_code",        IntegerType(), True),
    StructField("type",              IntegerType(), True),
    StructField("area",              IntegerType(), True),
    StructField("comment",           StringType(),  True),
    StructField("edits_pending",     IntegerType(), True),
    StructField("last_updated",      StringType(),  True),
    StructField("ended",             StringType(),  True),
])
genre_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("gid",           StringType(),  True),
    StructField("name",          StringType(),  True),
    StructField("comment",       StringType(),  True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
])
area_schema = StructType([
    StructField("id",               IntegerType(), True),
    StructField("gid",              StringType(),  True),
    StructField("name",             StringType(),  True),
    StructField("type",             IntegerType(), True),
    StructField("edits_pending",    IntegerType(), True),
    StructField("last_updated",     StringType(),  True),
    StructField("begin_date_year",  IntegerType(), True),
    StructField("begin_date_month", IntegerType(), True),
    StructField("begin_date_day",   IntegerType(), True),
    StructField("end_date_year",    IntegerType(), True),
    StructField("end_date_month",   IntegerType(), True),
    StructField("end_date_day",     IntegerType(), True),
    StructField("ended",            StringType(),  True),
    StructField("comment",          StringType(),  True),
])
tag_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("name",          StringType(),  True),
    StructField("ref_count",     IntegerType(), True),
])
gender_schema = StructType([
    StructField("id",   IntegerType(), True),
    StructField("name", StringType(),  True),
])
release_group_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("gid",           StringType(),  True),
    StructField("title",         StringType(),  True),
    StructField("artist_credit", IntegerType(), True),
    StructField("type",          IntegerType(), True),
    StructField("comment",       StringType(),  True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
])
l_artist_label_schema = StructType([
    StructField("id",             IntegerType(), True),
    StructField("link",           IntegerType(), True),
    StructField("entity0",        IntegerType(), True),  # artist
    StructField("entity1",        IntegerType(), True),  # label
    StructField("edits_pending",  IntegerType(), True),
    StructField("last_updated",   StringType(),  True),
    StructField("link_order",     IntegerType(), True),
    StructField("entity0_credit", StringType(),  True),
    StructField("entity1_credit", StringType(),  True),
])
l_artist_release_group_schema = StructType([
    StructField("id",             IntegerType(), True),
    StructField("link",           IntegerType(), True),
    StructField("entity0",        IntegerType(), True),  # artist
    StructField("entity1",        IntegerType(), True),  # release_group
    StructField("edits_pending",  IntegerType(), True),
    StructField("last_updated",   StringType(),  True),
    StructField("link_order",     IntegerType(), True),
    StructField("entity0_credit", StringType(),  True),
    StructField("entity1_credit", StringType(),  True),
])
label_tag_schema = StructType([
    StructField("label",        IntegerType(), True),
    StructField("tag",          IntegerType(), True),
    StructField("count",        IntegerType(), True),
    StructField("last_updated", StringType(),  True),
])
l_artist_genre_schema = StructType([
    StructField("id",             IntegerType(), True),
    StructField("link",           IntegerType(), True),
    StructField("entity0",        IntegerType(), True),  # artist
    StructField("entity1",        IntegerType(), True),  # genre
    StructField("edits_pending",  IntegerType(), True),
    StructField("last_updated",   StringType(),  True),
    StructField("link_order",     IntegerType(), True),
    StructField("entity0_credit", StringType(),  True),
    StructField("entity1_credit", StringType(),  True),
])
l_artist_artist_schema = StructType([
    StructField("id",      IntegerType(), True),
    StructField("link",    IntegerType(), True),
    StructField("entity0", IntegerType(), True),
    StructField("entity1", IntegerType(), True),
])
release_group_tag_schema = StructType([
    StructField("release_group", IntegerType(), True),
    StructField("tag",           IntegerType(), True),
    StructField("count",         IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
])
artist_tag_schema = StructType([
    StructField("artist",       IntegerType(), True),
    StructField("tag",          IntegerType(), True),
    StructField("count",        IntegerType(), True),
    StructField("last_updated", StringType(),  True),
])
artist_credit_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("name",          StringType(),  True),
    StructField("artist_count",  IntegerType(), True),
    StructField("ref_count",     IntegerType(), True),
    StructField("created",       StringType(),  True),
    StructField("edits_pending", IntegerType(), True),
    StructField("gid",           StringType(),  True),
])
artist_credit_name_schema = StructType([
    StructField("artist_credit", IntegerType(), True),
    StructField("position",      IntegerType(), True),
    StructField("artist",        IntegerType(), True),
    StructField("name",          StringType(),  True),
    StructField("join_phrase",   StringType(),  True),
])
l_artist_instrument_schema = StructType([
    StructField("id",             IntegerType(), True),
    StructField("link",           IntegerType(), True),
    StructField("entity0",        IntegerType(), True),  # artist
    StructField("entity1",        IntegerType(), True),  # instrument
    StructField("edits_pending",  IntegerType(), True),
    StructField("last_updated",   StringType(),  True),
    StructField("link_order",     IntegerType(), True),
    StructField("entity0_credit", StringType(),  True),
    StructField("entity1_credit", StringType(),  True),
])
link_schema = StructType([
    StructField("id",                IntegerType(), True),
    StructField("link_type",         IntegerType(), True),
    StructField("begin_date_year",   IntegerType(), True),
    StructField("begin_date_month",  IntegerType(), True),
    StructField("begin_date_day",    IntegerType(), True),
    StructField("end_date_year",     IntegerType(), True),
    StructField("end_date_month",    IntegerType(), True),
    StructField("end_date_day",      IntegerType(), True),
    StructField("attribute_count",   IntegerType(), True),
    StructField("created",           StringType(),  True),
    StructField("ended",             StringType(),  True),
])
link_type_schema = StructType([
    StructField("id",                  IntegerType(), True),
    StructField("parent",              IntegerType(), True),
    StructField("child_order",         IntegerType(), True),
    StructField("gid",                 StringType(),  True),
    StructField("entity_type0",        StringType(),  True),
    StructField("entity_type1",        StringType(),  True),
    StructField("name",                StringType(),  True),
    StructField("description",         StringType(),  True),
    StructField("link_phrase",         StringType(),  True),
    StructField("reverse_link_phrase", StringType(),  True),
    StructField("long_link_phrase",    StringType(),  True),
    StructField("last_updated",        StringType(),  True),
    StructField("is_deprecated",       StringType(),  True),
    StructField("has_dates",           StringType(),  True),
    StructField("attribute_count",     IntegerType(), True),
    StructField("priority",            IntegerType(), True),
])
release_schema = StructType([
    StructField("id",              IntegerType(), True),
    StructField("gid",             StringType(),  True),
    StructField("name",            StringType(),  True),
    StructField("artist_credit",   IntegerType(), True),
    StructField("release_group",   IntegerType(), True),
    StructField("status",          IntegerType(), True),
    StructField("packaging",       IntegerType(), True),
    StructField("language",        IntegerType(), True),
    StructField("script",          IntegerType(), True),
    StructField("barcode",         StringType(),  True),
    StructField("comment",         StringType(),  True),
    StructField("edits_pending",   IntegerType(), True),
    StructField("quality",         IntegerType(), True),
    StructField("last_updated",    StringType(),  True),
])
release_label_schema = StructType([
    StructField("id",             IntegerType(), True),
    StructField("release",        IntegerType(), True),
    StructField("label",          IntegerType(), True),
    StructField("catalog_number", StringType(),  True),
    StructField("last_updated",   StringType(),  True),
])

schemas = {
    "artist": artist_schema,
    "instrument": instrument_schema,
    "label": label_schema,
    "genre": genre_schema,
    "area": area_schema,
    "tag": tag_schema,
    "gender": gender_schema,
    "release_group": release_group_schema,
    "l_artist_label": l_artist_label_schema,
    "l_artist_release_group": l_artist_release_group_schema,
    "label_tag": label_tag_schema,
    "l_artist_genre": l_artist_genre_schema,
    "l_artist_artist": l_artist_artist_schema,
    "release_group_tag": release_group_tag_schema,
    "artist_tag": artist_tag_schema,
    "artist_credit": artist_credit_schema,
    "artist_credit_name": artist_credit_name_schema,
    "l_artist_instrument": l_artist_instrument_schema,
    "link": link_schema,
    "link_type": link_type_schema,
    "release": release_schema,
    "release_label": release_label_schema,
}

**Table Details**

In [ ]:
dfs = {}

for table_name, schema in schemas.items():
    print(f"\n==============================")
    print(f"Loading table: {table_name}")
    print(f"==============================")

    df = (
        spark.read
        .option("sep", "\t")
        .option("nullValue", r"\N")
        .option("header", "false")
        .option("quote", "")
        .option("escape", "")
        .schema(schema)
        .csv(f"{MBDUMP}/{table_name}")
    )

    dfs[table_name] = df
    
   # row_count = df.count()
   # print(f"Total {table_name} rows: {row_count}")
   # duplicate_count = row_count - df.dropDuplicates().count()
   # print(f"Duplicate rows in {table_name}: {duplicate_count}")

    print("=== SCHEMA ===")
   # df.printSchema()

    print("=== PEEK ===")
    peek(df)

**Explore and Summarize Columns**

The below section creates a profiling loop that classifies columns by their role and then summarizes them. The MusicBrainz dataset has a lot of IDs and relationship table columns so we need to sift through it and figure out which numeric columns should be summarized like a continuous variable verses an ID. The loop reports nulls, distinct counts, duplicates, categorical frequencies, numeric summaries, and foreign key uniqueness depending on the column type. 

We started by idetifying the primary keys, global ID, forgein key, date part, boolean, text descriptor and count measure clumns for each of the tables we chose.

We then created the function __get_column_role__ that classifies a given column's role/data type. This function is executed in the for loop which loops through each table's columns to summarize them based on their role which was assigned in the __get_column_role__ function. 

- ID (or id-ish) columns like id, gid, entity0, and entity1, the loop returns missing values, distinct counts, and if foreign keys match the referenced table (not sure if this is actually working how we want it to so take this part with a grain of salt). Not showing frequency distributions bc most ID values are unique and a top 10 count table wouldnt be value add

- Categorical columns are summarized using value counts and percentages to show which values are most common and whether the distribution is balanced or skewed

- Numeric/count columns are summarized with count, mean, standard deviation, min, quartiles, and max

- High-cardinality text columns, like names or UUIDs, get summarized with distinct counts and length stats instead of value counts bc there are way too many unique values

In [ ]:
primary_key_cols = {"id"}
global_id_cols = {"gid"}
foreign_key_map = {
    "artist": {
        "type":       ("artist_type", "id"),
        "area":       ("area", "id"),
        "gender":     ("gender", "id"),
        "begin_area": ("area", "id"),
        "end_area":   ("area", "id"),
    },
    "instrument": {
        "type": ("instrument_type", "id"),
    },
    "label": {
        "type": ("label_type", "id"),
        "area": ("area", "id"),
    },
    "release_group": {
        "type":          ("release_group_primary_type", "id"),
        "artist_credit": ("artist_credit", "id"),
    },
    "release": {
        "artist_credit": ("artist_credit", "id"),
        "release_group": ("release_group", "id"),
        "status":        ("release_status", "id"),
        "packaging":     ("release_packaging", "id"),
        "language":      ("language", "id"),
        "script":        ("script", "id"),
    },
    "release_label": {
        "release": ("release", "id"),
        "label":   ("label", "id"),
    },
    "artist_credit_name": {
        "artist_credit": ("artist_credit", "id"),
        "artist":        ("artist", "id"),
    },
    "label_tag": {
        "label": ("label", "id"),
        "tag":   ("tag", "id"),
    },
    "artist_tag": {
        "artist": ("artist", "id"),
        "tag":    ("tag", "id"),
    },
    "release_group_tag": {
        "release_group": ("release_group", "id"),
        "tag":           ("tag", "id"),
    },
    "l_artist_genre": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("genre", "id"),
    },
    "l_release_group_genre": {
        "link":    ("link", "id"),
        "entity0": ("release_group", "id"),
        "entity1": ("genre", "id"),
    },
    "l_artist_label": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("label", "id"),
    },
    "l_artist_release_group": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("release_group", "id"),
    },
    "l_artist_artist": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("artist", "id"),
    },
    "l_artist_instrument": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("instrument", "id"),
    },
    "link": {
        "link_type": ("link_type", "id"),
    },
    "link_type": {
        "parent": ("link_type", "id"),
    },
}
date_part_cols = {"begin_date_year", "begin_date_month", "begin_date_day", "end_date_year", "end_date_month", "end_date_day"}
boolean_like_cols = {"ended"}
text_descriptor_cols = {"name", "sort_name", "comment", "join_phrase", "last_updated"}
count_measure_cols = {"count", "ref_count", "artist_count", "edits_pending", "position"}


In [ ]:
def get_column_role(table_name, column_name, row_count, distinct_count):
    distinct_ratio = distinct_count / row_count if row_count else 0

    table_fk = foreign_key_map.get(table_name, {})

    rules = [
                (column_name in primary_key_cols, "primary_key"),
                (column_name in global_id_cols, "global_identifier"),
                (column_name in table_fk, "foreign_key"),
                (column_name in date_part_cols, "date_part"),
                (column_name in boolean_like_cols, "boolean_like"),
                (column_name in count_measure_cols, "count_measure"),
            ]

    for condition, role in rules:
        if condition:
            return role

    if column_name in text_descriptor_cols:
        return "high_cardinality_text" if distinct_ratio > 0.9 else "categorical_text"

    if distinct_ratio > 0.9:
        return "identifier_like"

    if distinct_count <= 25:
        return "categorical"

    return "numeric_or_high_cardinality"

In [ ]:
profile_rows = []

for table_name, df in dfs.items():
    print(f"\n\n==============================")
    print(f"TABLE: {table_name}")
    print(f"==============================")

    for field in df.schema.fields:
        column_name = field.name
        dtype = field.dataType.simpleString()

        missing_count = df.filter(col(column_name).isNull()).count()
        distinct_count = df.select(approx_count_distinct(col(column_name))).first()[0]
        distinct_ratio = distinct_count / row_count if row_count else 0

        role = get_column_role(table_name, column_name, row_count, distinct_count)

        print(f"\n--- {column_name} ({dtype}) ---")
        print(f"Role: {role}")
        print(f"Missing: {missing_count}")
        print(f"Distinct: {distinct_count}")
        print(f"Distinct ratio: {distinct_ratio:.4f}")

        summary = {
            "table": table_name,
            "column": column_name,
            "dtype": dtype,
            "role": role,
            "row_count": row_count,
            "missing_count": missing_count,
            "missing_pct": missing_count / row_count if row_count else None,
            "distinct_count": distinct_count,
            "distinct_ratio": distinct_ratio,
            "duplicate_rows_in_table": duplicate_count,
        }

        if role in ["primary_key", "global_identifier", "identifier_like"]:
            print("Summary: identifier column; frequency distributions are not meaningful.")

            if isinstance(field.dataType, StringType):
                df.select(
                    spark_min(length(col(column_name))).alias("min_length"),
                    spark_max(length(col(column_name))).alias("max_length")
                ).show()

        elif role == "foreign_key":
            summary["references"] = str(foreign_key_map[table_name].get(column_name))

        elif role in ["categorical", "categorical_text", "boolean_like", "date_part"]:
            print("Top values:")
            (
                df.groupBy(column_name)
                .count()
                .withColumn("pct", col("count") / row_count)
                .orderBy(col("count").desc())
                .show(10, truncate=False)
            )

        elif role in ["count_measure", "numeric_or_high_cardinality"]:
            print("Numeric summary:")
            df.select(column_name).summary(
                "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
            ).show()

        elif role == "high_cardinality_text":
            print("Summary: high-cardinality text; showing length stats instead of value counts.")
            df.select(
                spark_min(length(col(column_name))).alias("min_length"),
                spark_max(length(col(column_name))).alias("max_length"),
                avg(length(col(column_name))).alias("avg_length")
            ).show()

        profile_rows.append(summary)

profile_df = spark.createDataFrame(profile_rows)
profile_df.show(200, truncate=False)

### 4. Data Plots

**Artist Gender Pie Chart**

The following pie chart is created to illustrate the distribution of artists by gender in the dataset. We joined the artist and gender tables together using the gender foreign key and then counting the number of artists per gender group. In this pie chart, we see that a majority(approximately 76.5%) of artists are male while the second largest group is female(approximately 23%). There are a few small groups of gender classification that make up the remaining 0.5% but overall we can see that our data skews heavily to one gender.

In [ ]:
# garbage collection (maybe overkill but we can adjust this later)
del (
    total_cores, total_memory_gb, driver_memory_gb, executor_instances,
    raw_executor_memory, executor_memory_gb, executor_overhead_gb,
    sc, url, response, executors, spark_df,
    artist_schema, instrument_schema, label_schema, genre_schema, area_schema,
    tag_schema, gender_schema, release_group_schema, l_artist_label_schema,
    l_artist_release_group_schema, label_tag_schema, l_artist_genre_schema,
    l_artist_artist_schema, release_group_tag_schema, artist_tag_schema,
    artist_credit_schema, artist_credit_name_schema, l_artist_instrument_schema,
    schemas,
    primary_key_cols, global_id_cols, foreign_key_map, date_part_cols,
    boolean_like_cols, text_descriptor_cols, count_measure_cols,
    get_column_role, profile_rows, profile_df,
    df, table_name, field, column_name, dtype, missing_count,
    distinct_count, distinct_ratio, role, summary, row_count, duplicate_count,
    MBDUMP
)
import gc; gc.collect()

In [ ]:
artistdf = dfs["artist"]
genderdf = dfs["gender"]

In [ ]:
artists_by_gender = artistdf.join(
    genderdf,
    artistdf.gender == genderdf.id
).groupBy(
    genderdf.name.alias("gender")
).agg(
    count("*").alias("count")
).orderBy(
    "count", ascending=False
)
artists_by_gender.show()

In [ ]:
artist_gender = artists_by_gender.toPandas()

In [ ]:
plt.figure(figsize=(8,5))
plt.pie(artist_gender["count"], labels = artist_gender["gender"], autopct ='%1.1f%%')
plt.title("Distribution of Artists by Gender")
plt.tight_layout()
plt.show()

**Instrument Type by Count Bar Chart**

This bar chart displays the distribution of instrument types used by artists. Using the instrument table we grouped by the type column and then counted each type. The chart shows that the most common instrument type is 2 followed by 1 and 3, while types 4,5,6,7 were less common having appeared less than 50 times each. These type values represent numeric identifiers, which could be further joined with the instrument_type table to obtain more descriptive labels for each category.

In [ ]:
instrument_counts = dfs["instrument"].groupBy("type") \
    .agg(count("*").alias("count")) \
    .orderBy("count", ascending=False) \

instrument_dist = instrument_counts.toPandas()

plt.figure(figsize=(8,5))
plt.bar(instrument_dist["type"].astype(str), instrument_dist["count"])
plt.title("Instruments Type Used By Count")
plt.xlabel("Instrument Type")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

**Release Group Tag Bar Chart**

This horizontal bar chart uses the release_group_tag and tag tables to identify the top used tags for music. We did this to see what are the most common music genres tagged when music is released. The chart shows that electronic is the most common with over 500000 tags followed by rock. These two are the most common by far but interestingly the variations of rock like pop rock, indie rock, and alternative rock were some of the least used tags. This suggests that broader genre labels are used more often versus specialized classifications.

In [ ]:
release_tags = dfs["release_group_tag"].join(
    dfs["tag"],
    dfs["release_group_tag"].tag == dfs["tag"].id
).groupBy(
    dfs["tag"].name.alias("tag")
).agg(
    count("*").alias("count")
).orderBy(
    "count", ascending=False
).limit(15)

release_tags.show()

In [ ]:
release_tags_pd = release_tags.toPandas()

plt.figure(figsize=(10,6))
plt.barh(release_tags_pd["tag"], release_tags_pd["count"])
plt.title("Top 15 Tags for Release Groups")
plt.xlabel("Number of Release Group Tags")
plt.ylabel("Tag")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

**Artist Label Count Bar Chart**

This bar chart shows the top 15 record labels with the highest number of artists linked to them. We joined the l_artist_label and label tables in order to count the number of artist per label.In the chart we see that "Audio Network" has the highest amount of artists linked to them by a wide margin. There are also only 7 record labels with 100 or more artists linked to them.Overall, the distribution suggests that a small number of labels are connected to many artists following a similar pattern to the other data visualized.

In [ ]:
l_artist_label = dfs["l_artist_label"].select(
    col("entity1").alias("label_id")
)

labels = dfs["label"].select(
    col("id").alias("label_id"),
    col("name").alias("label")
)

label_artist_counts = (
    l_artist_label
    .join(broadcast(labels), on="label_id", how="inner")
    .groupBy("label")
    .agg(count("*").alias("artist_links"))
    .orderBy(col("artist_links").desc())
    .limit(15)
)

label_artist_counts.show(truncate=False)

In [ ]:
label_artist_pd = label_artist_counts.toPandas()

plt.figure(figsize=(10,6))
plt.barh(label_artist_pd["label"], label_artist_pd["artist_links"])
plt.title("Top 15 Labels by Number of Linked Artists")
plt.xlabel("Number of Artist-Label Links")
plt.ylabel("Label")
plt.tight_layout()
plt.show()

**Artist Credit Histogram**

This histogram shows the distribution of the number of artists associated with each artist credit. We grouped the artist_credit_name table by artist_credit and counted how many artists were linked to each credit. The chart shows that the vast majority of artist credits involve a single artist, with significantly fewer credits involving multiple artists. This indicates that the dataset has far more solo artist credits than collaborations so collaborations will be rare in the dataset.

In [ ]:
credit_sizes = dfs["artist_credit_name"].groupBy("artist_credit") \
    .agg(count("*").alias("num_artists"))
credit_sizes.show(5)

In [ ]:
cred = credit_sizes.toPandas()

plt.figure(figsize=(8,5))

plt.hist(cred["num_artists"], bins=20)

plt.title("Distribution of Artists per Credit")
plt.xlabel("Number of Artists in Credit")
plt.ylabel("Count")

plt.tight_layout()
plt.show()

**Artist By Area Bar Chart**

The final bar plot was created to highlight the number of artists per country in the dataset. We focused on the top 15 countries and did so by joining the area and artist table. This plot shows that there is a large faction of artists that are from the United States. This shows that the dataset is heavily skewed to the US but there are other countries with a high amount of artists like Japan, Germany, and the United Kingdom.

In [ ]:
artist_areas = dfs["artist"].join(
    dfs["area"],
    dfs["artist"].area == dfs["area"].id
).groupBy(
    dfs["area"].name.alias("area")
).agg(
    count("*").alias("count")
).orderBy(
    "count", ascending=False
).limit(15)

In [ ]:
art_area = artist_areas.toPandas()

plt.figure(figsize=(10,6))
plt.barh(art_area["area"], art_area["count"])

plt.title("Top Artist Areas")
plt.xlabel("Number of Artists")
plt.ylabel("Area")
plt.tight_layout()
plt.show()

### Preprocessing Plan

Our first milestone will be to build a K-Nearest Neighbors which groups music based on its "DNA" (instruments, genres, labels, etc), so the first step will be to determine precisely which variables constitute part of music's DNA, and determining their relative importance. The next step will be to strip away unnecessary information, such as columns and tables we won't need (for example: "description" in the instruments table, since it contains natural language, will not be helpful for this problem). 

The data has a high volume of nulls. Depending on the data type these will need to be dealt with differently - for example, date columns might be left as nulls because imputing an "average" date probably doesn't make any sense. "Type" columns (which indicate something else for each table, but can refer to things like release type: original, bootleg, reissue, etc) have their most common entry set to 1, so nulls in that variable will likely be set to 1. There are also lots of entries in which most data is null - since the dataset is so large, some sets of rows might be filtered out completely.

We will primarily use SQL for preparing the data that we will operate on, then switch over to Python for filling nulls and cleaning. Some Spark transformations we may use include join, groupBy, collect_list, StringIndexer, OneHotEncoder, CountVectorizer, VectorAssembler, Normalizer, and filter.

***

### UPDATED APPROACH:
After some testing I made some interesting findings. Apparently, linking "genre" and "artist" is tricky because the linking table l_artist_genre is practically empty, meaning when joining that way the resulting table only had about 400 rows. A workaround is to utilize tags- there is already a lot of overlap between genres and tags, and using tags as a link gives a much bigger dataset (1.5M rows!). SO- I'm creating a table with "release_name", "artist_name", "area_name", "label_name" and "tag_name" (contains multiple tags). Of the tags, the FIRST is taken and used as "genre_name" (they are already frequently the same, or at least similar), and the rest are squished into a vector stored under "tags".

So essentially, we are predicting the target variable "genre_name", which is actually just the first tag, based on "artist_name", "label_name", "area_name", and "tags", where "release_name" acts as an anchor connecting the features together (we are not training on it!!)

After this I use CountVectorizor to convert the tag list into a sparse binary vector, StringIndexer to convert genre/artist/area/label into numbers, and VectorAssembler to turn all of these into a single "features" vector

### BEFORE PROCEEDING
I still need to do some imputing logic for dealing with nulls and we need to maybe think about feature engineering (if it is even possible / makes sense given the type of data). It's late and brain not working too good but I will ponder this tomorrow. Once that is done we should be ready for model training 🤞

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.ml.feature import CountVectorizer, StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline


# pull in needed tables
release_df = dfs["release"].select(
    F.col("id").alias("release_id"),
    F.col("name").alias("release_name"),
    F.col("artist_credit").alias("release_artist_credit"),
    F.col("release_group").alias("release_release_group"),
)

acn_df = dfs["artist_credit_name"].select(
    F.col("artist_credit").alias("acn_artist_credit"),
    F.col("artist").alias("acn_artist_id"),
)

artist_df = dfs["artist"].select(
    F.col("id").alias("artist_id"),
    F.col("area").alias("artist_area_id"),
    F.col("gender").alias("artist_gender_id"),
)

gender_df = dfs["gender"].select(
    F.col("id").alias("gender_id"),
    F.col("name").alias("artist_gender"),
)

area_df = dfs["area"].select(
    F.col("id").alias("area_id"),
    F.col("name").alias("area_name"),
)

rl_df = dfs["release_label"].select(
    F.col("release").alias("rl_release_id"),
    F.col("label").alias("rl_label_id"),
)

label_df = dfs["label"].select(
    F.col("id").alias("label_id"),
    F.col("name").alias("label_name"),
)

rgt_df = dfs["release_group_tag"].select(
    F.col("release_group").alias("rgt_release_group"),
    F.col("tag").alias("rgt_tag_id"),
    F.col("count").alias("tag_count"),
)

tag_df = dfs["tag"].select(
    F.col("id").alias("tag_id"),
    F.col("name").alias("tag_name"),
)

# build table
model_df = (
    release_df
    # dropping releases with no artist
    .join(acn_df, release_df["release_artist_credit"] == acn_df["acn_artist_credit"], "inner")
    .join(artist_df, acn_df["acn_artist_id"] == artist_df["artist_id"], "inner")
    # gender (left: bands and some artists have no gender set)
    .join(gender_df, artist_df["artist_gender_id"] == gender_df["gender_id"], "left")
    .join(area_df, artist_df["artist_area_id"] == area_df["area_id"], "left")
    .join(rl_df, release_df["release_id"] == rl_df["rl_release_id"], "left")
    .join(label_df, rl_df["rl_label_id"] == label_df["label_id"], "left")
    .join(rgt_df, release_df["release_release_group"] == rgt_df["rgt_release_group"], "left")
    .join(tag_df, rgt_df["rgt_tag_id"] == tag_df["tag_id"], "left")
    .select(
        "release_name",
        "artist_gender",
        "area_name",
        "label_name",
        "tag_name",
        "tag_count",
    )
)

# ranking tags
tag_window = Window.partitionBy("release_name").orderBy(F.col("tag_count").desc_nulls_last())
ranked_df = model_df.withColumn("tag_rank", F.row_number().over(tag_window))

# aggregating on release_name, assigning top tag to "genre_name", storing the rest in "tags"
aggregated_df = (
    ranked_df
    .groupBy("release_name")
    .agg(
        F.first("artist_gender", ignorenulls=True).alias("artist_gender"),
        F.first("area_name",     ignorenulls=True).alias("area_name"),
        F.first("label_name",    ignorenulls=True).alias("label_name"),
        F.max(F.when(F.col("tag_rank") == 1, F.col("tag_name"))).alias("genre_name"),
        F.collect_set(
            F.when(F.col("tag_rank") > 1, F.col("tag_name"))
        ).alias("tags"),
    )
    .filter(F.col("genre_name").isNotNull())
)

# filter to recognized real genre names only (case-insensitive)
valid_genres_pd = pd.read_csv("real_music_genres.csv")
valid_genres = set(valid_genres_pd["Genre"].dropna().str.strip().str.lower())
aggregated_df = aggregated_df.filter(F.lower(F.col("genre_name")).isin(list(valid_genres)))

# map each genre to one of 19 broad buckets
buckets_pd = pd.read_csv("genre_buckets_wide.csv")
genre_to_bucket = {}
for bucket in buckets_pd.columns:
    for genre in buckets_pd[bucket].dropna():
        genre_clean = genre.strip().lower()
        if genre_clean:
            genre_to_bucket[genre_clean] = bucket

bucket_lookup_df = spark.createDataFrame(
    [(k, v) for k, v in genre_to_bucket.items()],
    ["genre_key", "bucket_name"],
)

aggregated_df = (
    aggregated_df
    .join(F.broadcast(bucket_lookup_df),
          F.lower(F.col("genre_name")) == F.col("genre_key"), "left")
    .drop("genre_name", "genre_key")
    .withColumnRenamed("bucket_name", "genre_name")
    .filter(F.col("genre_name").isNotNull())
)

print(f"\n==============================")
print(f"FINAL TABLE BEFORE ENCODING")
print(f"==============================")
aggregated_df.printSchema()
aggregated_df = aggregated_df.cache()
print("Row count:", aggregated_df.count())
peek(aggregated_df)

# using CountVectorizer to convert tag list into a sparse binary vector
cv = CountVectorizer(inputCol="tags", outputCol="tag_vector", minDF=2.0)
cv_model = cv.fit(aggregated_df)
vectorized_df = cv_model.transform(aggregated_df)

print(f"\n==============================")
print(f"FINAL TABLE AFTER ENCODING")
print(f"==============================")
print("Tag vocabulary size:", len(cv_model.vocabulary))
print("Label count:", aggregated_df.select("label_name").distinct().count())

genre_list = sorted([row[0] for row in aggregated_df.select("genre_name").distinct().collect()])
print("Genre count:", len(genre_list))
print("Genres:", genre_list)

# using StringIndexer to convert genre/gender/area/label into a numeric index
genre_indexer  = StringIndexer(inputCol="genre_name",    outputCol="genre_index",  handleInvalid="keep")
gender_indexer = StringIndexer(inputCol="artist_gender", outputCol="gender_index", handleInvalid="keep")
area_indexer   = StringIndexer(inputCol="area_name",     outputCol="area_index",   handleInvalid="keep")
label_indexer  = StringIndexer(inputCol="label_name",    outputCol="label_index",  handleInvalid="keep")

# putting all features into a single "features" vector using VectorAssembler
assembler = VectorAssembler(
    inputCols=["gender_index", "area_index", "label_index", "tag_vector"],
    outputCol="features",
    handleInvalid="keep",
)

# putting it all together
pipeline = Pipeline(stages=[
    genre_indexer, gender_indexer, area_indexer, label_indexer, assembler,
])

pipeline_model = pipeline.fit(vectorized_df)
final_df = pipeline_model.transform(vectorized_df)

final_df = final_df.select("release_name", "genre_index", "features").cache()
print("Row count:", final_df.count())
peek(final_df)

## RF Model 1

In [ ]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [ ]:
model_input = final_df.withColumnRenamed("genre_index", "label")
train_df, test_df = model_input.randomSplit([0.8, 0.2], seed=21) # split 80/20

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=50,
    maxDepth=8,
    maxBins=64,
    seed=21)

rf_model = rf.fit(train_df)

In [ ]:
predictions = rf_model.transform(test_df)
#check f1 for precison and recall
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1")
#percent of correct;y chosen labels
accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy")

In [ ]:
print("Random Forest F1:", f1_eval.evaluate(predictions))
print("Random Forest Accuracy:", accuracy_eval.evaluate(predictions))

predictions.select(
    "release_name",
    "label",
    "prediction",
    "probability"
).show(20, truncate=False)